# Encoder–decoder Transformer for legal continuation

## 1. Hugging Face Login

In [1]:
%pip install -q huggingface_hub ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 3.5 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 9. Define The Model

In [5]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class TransformerConfig:
    vocab_size: int
    pad_id: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    n_layers: int = 6
    dropout: float = 0.1
    max_src_len: int = 2048
    max_tgt_len: int = 2048
    pos_emb_type: str = "rope"
    rope_base: float = 10000.0


class EmbeddingDropout(nn.Module):
    # Preserve the embedding dropout and configured length limit without adding PE.
    def __init__(self, max_len, dropout):
        super().__init__()
        self.max_len = max_len
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        if x.size(1) > self.max_len:
            raise ValueError('Sequence exceeds configured context length')
        return self.dropout(x)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        if head_dim % 2:
            raise ValueError('RoPE requires an even head dimension')
        inv_freq = base ** (
            -torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
        )
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    def forward(self, x):
        # x: [batch, heads, length, head_dim]. Split-half pairs: (i, i + d/2).
        # Generation recomputes the full decoder sequence, so positions start at 0.
        with torch.autocast(device_type=x.device.type, enabled=False):
            positions = torch.arange(x.size(2), device=x.device, dtype=torch.float32)
            angles = torch.outer(positions, self.inv_freq.float())
            cos = angles.cos()[None, None, :, :]
            sin = angles.sin()[None, None, :, :]
        cos, sin = cos.to(x.dtype), sin.to(x.dtype)
        first, second = x.chunk(2, dim=-1)
        return torch.cat((first * cos - second * sin,
                          first * sin + second * cos), dim=-1)


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, use_rope=False, rope_base=10000.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.rope = RotaryEmbedding(self.d_head, rope_base) if use_rope else None
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        # [batch, length, d_model] -> [batch, heads, length, d_head]
        b, length, _ = x.shape
        return x.reshape(b, length, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, query, key, value, allowed, return_attention=False):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        if self.rope is not None:
            q = self.rope(q)
            k = self.rope(k)
        # True means this key may be attended to, including the causal mask.
        weights = None
        if return_attention:
            # Keep explicit probabilities for attention visualizations.
            with torch.autocast(device_type=q.device.type, enabled=False):
                scores = (q.float() @ k.float().transpose(-2, -1)) / math.sqrt(self.d_head)
                scores = scores.masked_fill(~allowed, float('-inf'))
                weights = scores.softmax(dim=-1)
            context = self.dropout(weights.to(v.dtype)) @ v
        else:
            # Let PyTorch select a supported backend; dropout must be 0 at eval.
            context = F.scaled_dot_product_attention(
                q, k, v, attn_mask=allowed,
                dropout_p=self.dropout.p if self.training else 0.0,
                is_causal=False,  # Causality is already present in allowed.
            )
        b, _, length, _ = context.shape
        context = context.transpose(1, 2).contiguous().reshape(b, length, -1)
        output = self.out_proj(context)
        return output, weights.mean(dim=1) if return_attention else None

In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, source_mask):
        attended, _ = self.attention(x, x, x, source_mask)
        x = self.norm1(x + self.dropout(attended))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.cross_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=False)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.norm3 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, memory, target_mask, source_mask, return_attention=False):
        attended, _ = self.self_attention(x, x, x, target_mask)
        x = self.norm1(x + self.dropout(attended))
        attended, weights = self.cross_attention(x, memory, memory, source_mask,
                                                  return_attention)
        x = self.norm2(x + self.dropout(attended))
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x, weights

In [9]:
class LegalTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        if cfg.pos_emb_type != 'rope':
            raise ValueError('This architecture expects a RoPE configuration')
        self.cfg = cfg
        # Separate weights make the three roles easy to inspect.
        self.source_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.target_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.source_position = EmbeddingDropout(cfg.max_src_len, cfg.dropout)
        self.target_position = EmbeddingDropout(cfg.max_tgt_len, cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.output = nn.Linear(cfg.d_model, cfg.vocab_size)
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)
        with torch.no_grad():
            self.source_embedding.weight[cfg.pad_id].zero_()
            self.target_embedding.weight[cfg.pad_id].zero_()

    def encode(self, source_ids):
        source_mask = source_ids.ne(self.cfg.pad_id)[:, None, None, :]
        x = self.source_embedding(source_ids) * math.sqrt(self.cfg.d_model)
        x = self.source_position(x)
        for layer in self.encoder:
            x = layer(x, source_mask)
        return x, source_mask

    def decode(self, target_ids, memory, source_mask, return_attention=False):
        length = target_ids.size(1)
        causal = torch.ones(length, length, dtype=torch.bool, device=target_ids.device).tril()
        valid_keys = target_ids.ne(self.cfg.pad_id)[:, None, None, :]
        target_mask = valid_keys & causal[None, None, :, :]
        x = self.target_embedding(target_ids) * math.sqrt(self.cfg.d_model)
        x = self.target_position(x)
        weights = None
        for index, layer in enumerate(self.decoder):
            x, weights = layer(x, memory, target_mask, source_mask,
                               return_attention and index == len(self.decoder) - 1)
        logits = self.output(x)
        return (logits, weights) if return_attention else logits

    def forward(self, source_ids, target_ids, return_attention=False):
        memory, source_mask = self.encode(source_ids)
        return self.decode(target_ids, memory, source_mask, return_attention)

# **Phase 3**

In [10]:
import json
import math
import random
import zipfile
from pathlib import Path
from dataclasses import asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import (
    precision_recall_fscore_support,
    classification_report,
)

MODEL_DIR = Path("/kaggle/input/datasets/anilkumarpalli/encdec-model-files")

METHOD = "lorma_plus"  # "lora" or "lorma_plus"

USE_CONTEXT = True
USE_POSITION = False
USE_CLASS_WEIGHTS = False

MAX_LENGTH = 512
BATCH_SIZE = 4
EPOCHS = 10
LEARNING_RATE = 1e-4
PATIENCE = 3

RANK = 8
ALPHA = 16
SEED = 42

RUN_NAME = f"encdec_{METHOD}_mean_eos_position"
OUTPUT_DIR = Path("/kaggle/working/phase3") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Device:", device)
print("Method:", METHOD)
print("Output:", OUTPUT_DIR)

Device: cuda
Method: lora
Output: /kaggle/working/phase3/encdec_lora_mean_eos_position


In [11]:
from huggingface_hub import hf_hub_download

DATA_DIR = Path("/kaggle/working/capstone_data")

files = [
    "Train/CL_train.json",
    "Train/CL_dev.json",
    "Train/IT_train.json",
    "Train/IT_dev.json",
    "Train/label_vocab.json",
    "Test/CL_test.json",
    "Test/IT_test.json",
]

for filename in files:
    hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        filename=f"Phase_3_RR/{filename}",
        local_dir=str(DATA_DIR),
        token=globals().get("HF_TOKEN"),
    )

RR_DIR = DATA_DIR / "Phase_3_RR"


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

In [12]:
checkpoint = torch.load(
    MODEL_DIR / "model.pt",
    map_location="cpu",
    weights_only=True,
)

cfg = TransformerConfig(**checkpoint["model_config"])

phase1_model = LegalTransformer(cfg)

phase1_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

del checkpoint

MAX_LENGTH = min(
    MAX_LENGTH,
    cfg.max_src_len,
    cfg.max_tgt_len,
)

state = load_json(MODEL_DIR / "tokenizer.json")

tokenizer = BytelevelBPE(state["vocab_size"])
tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(state["vocab"])

tokenizer.stoi = {
    symbol: i
    for i, symbol in enumerate(tokenizer.vocab)
}

tokenizer.itos = {
    i: symbol
    for symbol, i in tokenizer.stoi.items()
}

tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]

tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}

tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

del state

PAD_ID = tokenizer.stoi["<pad>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]
UNK_ID = tokenizer.stoi["<unk>"]

if (
    len(tokenizer.vocab) != cfg.vocab_size
    or PAD_ID != cfg.pad_id
):
    raise ValueError("Tokenizer does not match this checkpoint.")

print("Maximum sequence length:", MAX_LENGTH)

Maximum sequence length: 512


In [13]:
vocab = load_json(RR_DIR / "Train/label_vocab.json")

if isinstance(vocab, dict):
    for key in ("label2id", "id2label", "labels"):
        if key in vocab and isinstance(vocab[key], (dict, list)):
            vocab = vocab[key]
            break

if isinstance(vocab, list):
    label_names = vocab
elif all(str(key).isdigit() for key in vocab):
    label_names = [
        vocab[key]
        for key in sorted(vocab, key=lambda x: int(x))
    ]
else:
    label_names = sorted(
        vocab, key=lambda x: int(vocab[x])
    )

label2id = {
    label: i for i, label in enumerate(label_names)
}
id2label = dict(enumerate(label_names))
NUM_LABELS = len(label_names)


def read_documents(path, domain, labeled=True):
    raw = load_json(path)

    if isinstance(raw, dict) and "id" in raw and "text" in raw:
        records = [raw]

    elif isinstance(raw, dict):
        records = []

        for doc_id, record in raw.items():
            if isinstance(record, list):
                record = {"text": record}

            records.append({**record, "id": doc_id})

    else:
        records = raw

    documents = []
    seen = set()

    for record in records:
        doc_id = str(record["id"])
        sentences = record["text"]
        labels = record["labels"] if labeled else None

        if doc_id in seen:
            raise ValueError(f"Duplicate document ID: {doc_id}")
        seen.add(doc_id)

        if (
            not isinstance(sentences, list)
            or any(not isinstance(s, str) for s in sentences)
        ):
            raise ValueError("Expected a list of sentence strings.")

        if labeled:
            if len(sentences) != len(labels):
                raise ValueError(f"Sentence-label mismatch: {doc_id}")

            if any(label not in label2id for label in labels):
                raise ValueError(f"Unknown label in {doc_id}")

        documents.append({
            "id": doc_id,
            "domain": domain,
            "text": sentences,
            "labels": labels,
        })

    return documents


train_docs = []
dev_docs = []

for domain in ("CL", "IT"):
    train_docs += read_documents(
        RR_DIR / f"Train/{domain}_train.json",
        domain,
    )

    dev_docs += read_documents(
        RR_DIR / f"Train/{domain}_dev.json",
        domain,
    )

print("Labels:", label_names)
print("Training documents:", len(train_docs))
print("Development documents:", len(dev_docs))

Labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
Training documents: 80
Development documents: 10


In [14]:
previous_prefix = tokenizer.encode("Previous sentence:")
next_prefix = tokenizer.encode("Next sentence:")
target_prefix = tokenizer.encode("Target sentence:")

overhead = (
    4
    + len(previous_prefix)
    + len(next_prefix)
    + len(target_prefix)
)

target_capacity = MAX_LENGTH - overhead

if target_capacity < 1:
    raise ValueError("MAX_LENGTH is too small.")


def make_chunks(target, left, right):
    chunks = []

    for start in range(0, len(target), target_capacity):
        part = target[start:start + target_capacity]
        available = target_capacity - len(part)

        n_left = min(len(left), available // 2)
        n_right = min(len(right), available - n_left)
        n_left = min(len(left), available - n_right)

        left_part = left[-n_left:] if n_left else []
        right_part = right[:n_right]

        ids = (
            [BOS_ID]
            + previous_prefix + left_part + [EOS_ID]
            + next_prefix + right_part + [EOS_ID]
            + target_prefix + part + [EOS_ID]
        )

        chunks.append((ids, len(part)))

    return chunks


class RRDataset(Dataset):
    def __init__(self, documents):
        self.documents = documents
        self.tokens = []
        self.index = []

        for doc_index, document in enumerate(documents):
            encoded = []

            for sentence_index, sentence in enumerate(document["text"]):
                ids = tokenizer.encode(sentence)
                encoded.append(ids or [UNK_ID])

                self.index.append(
                    (doc_index, sentence_index)
                )

            self.tokens.append(encoded)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, index):
        doc_index, sentence_index = self.index[index]

        document = self.documents[doc_index]
        sentences = self.tokens[doc_index]

        left = []
        right = []

        if USE_CONTEXT:
            if sentence_index > 0:
                left = sentences[sentence_index - 1]

            if sentence_index + 1 < len(sentences):
                right = sentences[sentence_index + 1]

        chunks = make_chunks(
            sentences[sentence_index],
            left,
            right,
        )

        label = -100

        if document["labels"] is not None:
            label = label2id[
                document["labels"][sentence_index]
            ]

        position = sentence_index / max(
            1, len(sentences) - 1
        )

        return {
            "chunks": chunks,
            "label": label,
            "position": position,
            "meta": (
                document["domain"],
                document["id"],
                sentence_index,
            ),
        }


def collate_rr(examples):
    input_ids = []
    lengths = []
    owners = []
    weights = []
    positions = []

    for owner, example in enumerate(examples):
        for ids, target_length in example["chunks"]:
            input_ids.append(
                torch.tensor(ids, dtype=torch.long)
            )

            lengths.append(len(ids))
            owners.append(owner)
            weights.append(target_length)
            positions.append(example["position"])

    return {
        "input_ids": pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=PAD_ID,
        ),
        "lengths": torch.tensor(lengths, dtype=torch.long),
        "owners": torch.tensor(owners, dtype=torch.long),
        "weights": torch.tensor(weights, dtype=torch.float32),
        "sentence_positions": torch.tensor(
            positions, dtype=torch.float32
        ),
        "labels": torch.tensor(
            [example["label"] for example in examples],
            dtype=torch.long,
        ),
        "meta": [example["meta"] for example in examples],
    }


train_dataset = RRDataset(train_docs)
dev_dataset = RRDataset(dev_docs)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_rr,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
)

print("Training sentences:", len(train_dataset))
print("Development sentences:", len(dev_dataset))

Training sentences: 16845
Development sentences: 2142


In [15]:
class LoRA(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)
        self.scale = alpha / rank

        self.A = nn.Linear(
            base_layer.in_features, rank, bias=False
        )
        self.B = nn.Linear(
            rank, base_layer.out_features, bias=False
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return (
            self.base(x)
            + self.scale * self.B(self.A(x))
        )


class LoRMAPlus(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)
        self.scale = alpha / rank

        out_features = base_layer.out_features

        self.A = nn.Linear(
            out_features, rank, bias=False
        )
        self.B = nn.Linear(
            rank, out_features, bias=False
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        original = self.base(x)
        correction = self.B(self.A(original))

        return original + self.scale * correction

In [16]:
class EncoderDecoderRR(nn.Module):
    def __init__(self, pretrained_model, num_labels, use_position):
        super().__init__()

        self.cfg = pretrained_model.cfg
        self.use_position = use_position

        self.source_embedding = pretrained_model.source_embedding
        self.target_embedding = pretrained_model.target_embedding

        self.source_position = pretrained_model.source_position
        self.target_position = pretrained_model.target_position

        self.encoder = pretrained_model.encoder
        self.decoder = pretrained_model.decoder

        self.dropout = nn.Dropout(self.cfg.dropout)

        feature_size = (
            2 * self.cfg.d_model + int(use_position)
        )

        self.classifier = nn.Linear(
            feature_size, num_labels
        )

    def forward(
        self,
        input_ids,
        lengths,
        target_lengths,
        sentence_positions,
    ):
        length = input_ids.size(1)

        # Encoder: mask padding keys
        source_mask = input_ids.ne(
            self.cfg.pad_id
        )[:, None, None, :]

        memory = self.source_embedding(input_ids)
        memory = memory * math.sqrt(self.cfg.d_model)
        memory = self.source_position(memory)

        for layer in self.encoder:
            memory = layer(memory, source_mask)

        # Decoder: causal self-attention with padding masked
        causal = torch.ones(
            length,
            length,
            dtype=torch.bool,
            device=input_ids.device,
        ).tril()

        target_mask = (
            source_mask & causal[None, None, :, :]
        )

        x = self.target_embedding(input_ids)
        x = x * math.sqrt(self.cfg.d_model)
        x = self.target_position(x)

        for layer in self.decoder:
            x, _ = layer(
                x,
                memory,
                target_mask,
                source_mask,
            )

        x = x.float()

        # Final EOS representation
        rows = torch.arange(
            x.size(0), device=x.device
        )

        eos_positions = lengths - 1
        eos_vector = x[rows, eos_positions]

        # Mean over target tokens only
        target_starts = eos_positions - target_lengths

        token_positions = torch.arange(
            length, device=x.device
        ).unsqueeze(0)

        target_mask = (
            (token_positions >= target_starts.unsqueeze(1))
            & (token_positions < eos_positions.unsqueeze(1))
        )

        mask = target_mask.unsqueeze(-1).float()

        target_mean = (
            (x * mask).sum(dim=1)
            / mask.sum(dim=1).clamp_min(1)
        )

        features = torch.cat(
            [target_mean, eos_vector],
            dim=-1,
        )

        features = self.dropout(features)

        if self.use_position:
            features = torch.cat(
                [
                    features,
                    sentence_positions.float().unsqueeze(-1),
                ],
                dim=-1,
            )

        return self.classifier(features)

In [17]:
torch.manual_seed(SEED)

rr_model = EncoderDecoderRR(
    phase1_model,
    NUM_LABELS,
    use_position=USE_POSITION,
)

del phase1_model

rr_model.requires_grad_(False)

Adapter = {
    "lora": LoRA,
    "lorma_plus": LoRMAPlus,
}[METHOD]

ATTENTION_TARGETS = [
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
]

FFN_TARGETS = ["linear1", "linear2"]

attention_modules = [
    layer.attention
    for layer in rr_model.encoder
]

for layer in rr_model.decoder:
    attention_modules.extend([
        layer.self_attention,
        layer.cross_attention,
    ])

for attention in attention_modules:
    for name in ATTENTION_TARGETS:
        layer = getattr(attention, name)

        setattr(
            attention,
            name,
            Adapter(layer, RANK, ALPHA),
        )

for layer in list(rr_model.encoder) + list(rr_model.decoder):
    for name in FFN_TARGETS:
        base = getattr(layer.ffn, name)

        setattr(
            layer.ffn,
            name,
            Adapter(base, RANK, ALPHA),
        )

rr_model.classifier.requires_grad_(True)
rr_model = rr_model.to(device)

total = sum(
    p.numel() for p in rr_model.parameters()
)

trainable = sum(
    p.numel()
    for p in rr_model.parameters()
    if p.requires_grad
)

print("Method:", METHOD)
print("Classifier input:", rr_model.classifier.in_features)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Trainable percentage: {100 * trainable / total:.2f}%")

Method: lora
Classifier input: 1024
Total parameters: 65,676,301
Trainable parameters: 1,094,669
Trainable percentage: 1.67%


In [18]:
def move_batch(batch):
    return {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def get_sentence_logits(model, batch):
    chunk_logits = model(
        batch["input_ids"],
        batch["lengths"],
        batch["weights"].long(),
        batch["sentence_positions"],
    ).float()

    owners = batch["owners"]
    weights = batch["weights"]

    logits = chunk_logits.new_zeros(
        len(batch["meta"]), NUM_LABELS
    )

    logits = logits.index_add(
        0,
        owners,
        chunk_logits * weights[:, None],
    )

    totals = weights.new_zeros(len(batch["meta"]))
    totals = totals.index_add(0, owners, weights)

    return logits / totals[:, None]


counts = torch.zeros(NUM_LABELS)

for document in train_docs:
    for label in document["labels"]:
        counts[label2id[label]] += 1

class_weights = None

if USE_CLASS_WEIGHTS:
    class_weights = counts.clamp_min(1).rsqrt()
    class_weights /= class_weights.mean()
    class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

print(dict(zip(label_names, counts.int().tolist())))

{'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}


In [19]:
def macro_scores(gold, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        gold,
        predicted,
        labels=list(range(NUM_LABELS)),
        average="macro",
        zero_division=0,
    )

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


@torch.inference_mode()
def evaluate_rr(model, loader):
    model.eval()

    gold = []
    predicted = []
    domains = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_sentence_logits(model, batch)

        gold.extend(batch["labels"].tolist())
        predicted.extend(logits.argmax(-1).tolist())
        domains.extend(meta[0] for meta in batch["meta"])

    scores = macro_scores(gold, predicted)
    scores["domains"] = {}

    for domain in ("CL", "IT"):
        indices = [
            i for i, name in enumerate(domains)
            if name == domain
        ]

        if indices:
            scores["domains"][domain] = macro_scores(
                [gold[i] for i in indices],
                [predicted[i] for i in indices],
            )

    return scores, gold, predicted

In [20]:
trainable_parameters = [
    p for p in rr_model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16,
)

run_config = {
    "method": METHOD,
    "adapter_class": Adapter.__name__,
    "rank": RANK,
    "alpha": ALPHA,
    "attention_targets": ATTENTION_TARGETS,
    "ffn_targets": FFN_TARGETS,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "use_context": USE_CONTEXT,
    "use_position": USE_POSITION,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "pooling": "target_mean_plus_eos",
    "input_order": "previous_next_target",
    "seed": SEED,
}

(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2),
    encoding="utf-8",
)

history = []
best_f1 = -1.0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    rr_model.train()

    total_loss = 0.0
    total_weight = 0.0

    for step, batch in enumerate(train_loader, 1):
        batch = move_batch(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_sentence_logits(rr_model, batch)
            loss = criterion(logits, batch["labels"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_parameters, 1.0
        )

        scaler.step(optimizer)
        scaler.update()

        weight = (
            len(batch["meta"])
            if class_weights is None
            else class_weights[batch["labels"]].sum().item()
        )

        total_loss += loss.item() * weight
        total_weight += weight

        if step % 50 == 0:
            print(
                f"Epoch {epoch} | "
                f"Batch {step}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    scores, _, _ = evaluate_rr(rr_model, dev_loader)
    train_loss = total_loss / total_weight

    print(
        f"Epoch {epoch} | "
        f"Train loss: {train_loss:.4f} | "
        f"Dev Macro-F1: {scores['f1']:.4f}"
    )

    print("Domain scores:", scores["domains"])

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **scores,
    })

    (OUTPUT_DIR / "history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8",
    )

    if scores["f1"] > best_f1:
        best_f1 = scores["f1"]
        bad_epochs = 0

        torch.save(
            {
                "model_state_dict": {
                    name: tensor.detach().cpu()
                    for name, tensor in rr_model.state_dict().items()
                },
                "model_config": asdict(cfg),
                "run_config": run_config,
                "label2id": label2id,
                "epoch": epoch,
                "dev_scores": scores,
            },
            OUTPUT_DIR / "best_f1.pt",
        )

        print("Saved best checkpoint.")

    else:
        bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

Epoch 1 | Batch 50/4212 | Loss: 1.8489
Epoch 1 | Batch 100/4212 | Loss: 2.5450
Epoch 1 | Batch 150/4212 | Loss: 1.5164
Epoch 1 | Batch 200/4212 | Loss: 1.2046
Epoch 1 | Batch 250/4212 | Loss: 1.0821
Epoch 1 | Batch 300/4212 | Loss: 1.9317
Epoch 1 | Batch 350/4212 | Loss: 1.1246
Epoch 1 | Batch 400/4212 | Loss: 1.2370
Epoch 1 | Batch 450/4212 | Loss: 3.2816
Epoch 1 | Batch 500/4212 | Loss: 2.2805
Epoch 1 | Batch 550/4212 | Loss: 1.8167
Epoch 1 | Batch 600/4212 | Loss: 1.8328
Epoch 1 | Batch 650/4212 | Loss: 1.1142
Epoch 1 | Batch 700/4212 | Loss: 2.6521
Epoch 1 | Batch 750/4212 | Loss: 1.7646
Epoch 1 | Batch 800/4212 | Loss: 1.3808
Epoch 1 | Batch 850/4212 | Loss: 2.5834
Epoch 1 | Batch 900/4212 | Loss: 2.0789
Epoch 1 | Batch 950/4212 | Loss: 1.7144
Epoch 1 | Batch 1000/4212 | Loss: 1.8819
Epoch 1 | Batch 1050/4212 | Loss: 1.4496
Epoch 1 | Batch 1100/4212 | Loss: 3.3690
Epoch 1 | Batch 1150/4212 | Loss: 0.9172
Epoch 1 | Batch 1200/4212 | Loss: 1.2998
Epoch 1 | Batch 1250/4212 | Loss: 2.

In [21]:
checkpoint = torch.load(
    OUTPUT_DIR / "best_f1.pt",
    map_location="cpu",
    weights_only=True,
)

rr_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print("Best epoch:", checkpoint["epoch"])
del checkpoint

scores, gold, predicted = evaluate_rr(
    rr_model, dev_loader
)

print(
    classification_report(
        gold,
        predicted,
        labels=list(range(NUM_LABELS)),
        target_names=label_names,
        digits=4,
        zero_division=0,
    )
)

print("Domain scores:", scores["domains"])

Best epoch: 6
                        precision    recall  f1-score   support

                  Fact     0.5287    0.6216    0.5714       444
                 Issue     0.5294    0.6000    0.5625        15
    ArgumentPetitioner     0.6111    0.3025    0.4047       400
    ArgumentRespondent     0.5755    0.2243    0.3228       272
               Statute     0.7054    0.7583    0.7309       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.3754    0.4331    0.4022       254
PrecedentNotReliedUpon     0.2381    0.1000    0.1408        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.3793    0.1078    0.1679       102
    RatioOfTheDecision     0.3709    0.7022    0.4854       403
  RulingByPresentCourt     0.5000    0.3878    0.4368        49
                  None     0.2692    0.2121    0.2373        33

              accuracy                         0.4636      2142
             macro avg  

In [22]:
test_docs = []

for domain in ("CL", "IT"):
    test_docs += read_documents(
        RR_DIR / f"Test/{domain}_test.json",
        domain,
        labeled=False,
    )

test_dataset = RRDataset(test_docs)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
)


@torch.inference_mode()
def predict_rr(model, loader, documents):
    model.eval()

    predictions = {
        (doc["domain"], doc["id"]): [None] * len(doc["text"])
        for doc in documents
    }

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_sentence_logits(model, batch)

        predicted_ids = logits.argmax(-1).tolist()

        for meta, label_id in zip(batch["meta"], predicted_ids):
            domain, doc_id, sentence_index = meta

            predictions[(domain, doc_id)][sentence_index] = (
                id2label[label_id]
            )

    return predictions


predictions = predict_rr(
    rr_model, test_loader, test_docs
)

print("Predicted documents:", len(predictions))

Predicted documents: 10


In [23]:
submission = {}

for (domain, doc_id), labels in predictions.items():
    if doc_id in submission:
        raise ValueError(
            f"ID {doc_id} occurs in both domains. "
            "Check the official submission format."
        )

    if any(label is None for label in labels):
        raise ValueError(f"Missing predictions for {doc_id}")

    submission[doc_id] = labels

prediction_path = OUTPUT_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(
        submission,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

with zipfile.ZipFile(
    OUTPUT_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )

print("Saved:", OUTPUT_DIR / "submission.zip")

Saved: /kaggle/working/phase3/encdec_lora_mean_eos_position/submission.zip
